# 2. Feedforward neural network experiments

Run notebook 01 first. This notebook trains all four embeddings with three configurations: **12 runs**. It fits on the training split and selects checkpoints using **validation macro F1**, breaking ties with validation loss. Test predictions are reserved for notebook 04.


Each message becomes the **masked mean of its word vectors**, then one or two dense hidden layers, then a three-class softmax. Padding is excluded from the mean. This pooling choice is an implementation assumption because the paper does not specify it; it discards word order. Hidden widths 64/32, Adam (0.001), batch size 128, dropout 0.2, and frozen embeddings are also explicit assumptions.

The three hidden configurations mirror the activation/depth combinations in Table 5: ReLU; ReLU -> ReLU; ReLU -> sigmoid. Every model ends in softmax. Comparing the last two holds widths/depth constant and changes the second hidden activation; the one-layer model separately tests depth.


The default runs up to the paper's epoch budget with early stopping (25 epochs without improvement). Set `EARLY_STOPPING = False` to complete every epoch. The checkpoint still comes from the best validation epoch, which may differ from the paper's unspecified checkpoint policy. A single seed is used for readability; small score differences should be checked with multiple seeds before making strong claims. Training curves use batches during training (with dropout); validation curves use evaluation mode.

Dependencies are installed by notebook 01. If using a new Python environment, run its installation cell first. Leave the four notebooks together. Rerunning this notebook replaces this family's saved runs with the same names.


In [ ]:
from pathlib import Path
import copy
import json
import random
import time
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score
from IPython.display import display

ROOT = Path.cwd()
ARTIFACTS = ROOT / "artifacts"
assert (ARTIFACTS / "metadata.json").exists(), "Run notebook 01 in this folder first."
metadata = json.loads((ARTIFACTS / "metadata.json").read_text())
with np.load(ARTIFACTS / "encoded_data.npz") as saved:
    # Training notebooks deliberately do not load the test arrays.
    X_train, y_train = saved["X_train"], saved["y_train"]
    X_val, y_val = saved["X_validation"], saved["y_validation"]
with np.load(ARTIFACTS / "embeddings.npz") as saved:
    embeddings = {name: saved[name] for name in saved.files}

SEED = metadata["seed"]
BATCH_SIZE = 128
LEARNING_RATE = 0.001
EARLY_STOPPING = True
PATIENCE = 25
TRAINABLE_EMBEDDINGS = False  # fixed embeddings for a controlled comparison
DROPOUT = 0.2
# CUDA when available; CPU works too and is easiest to reproduce across machines.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(4, torch.get_num_threads()))
print("Device:", DEVICE, "| Word2Vec source:", metadata["word2vec_source"])


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_loader(X, y, shuffle=False):
    dataset = TensorDataset(torch.tensor(X, dtype=torch.long),
                            torch.tensor(y, dtype=torch.long))
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
                      generator=generator, num_workers=0)


# Both alternatives return log probabilities for the same categorical loss.
def log_probabilities(logits, output_activation):
    if output_activation == "softmax":
        return torch.log_softmax(logits, dim=1)
    # Normalize sigmoid scores to sum to 1 for mutually exclusive classes.
    log_scores = torch.nn.functional.logsigmoid(logits)
    return log_scores - torch.logsumexp(log_scores, dim=1, keepdim=True)


## Model definition


In [ ]:
class FeedforwardNN(nn.Module):
    def __init__(self, matrix, config):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(matrix), freeze=not TRAINABLE_EMBEDDINGS, padding_idx=0)
        layers = []
        width = matrix.shape[1]
        for hidden_width, activation in zip(config["hidden_sizes"], config["activations"]):
            layers.append(nn.Linear(width, hidden_width))
            layers.append(nn.ReLU() if activation == "relu" else nn.Sigmoid())
            layers.append(nn.Dropout(DROPOUT))
            width = hidden_width
        layers.append(nn.Linear(width, 3))
        self.classifier = nn.Sequential(*layers)

    def forward(self, X):
        mask = (X != 0).unsqueeze(-1)
        vectors = self.embedding(X)
        pooled = (vectors * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return log_probabilities(self.classifier(pooled), "softmax")


## Experiment settings


In [ ]:
EPOCHS = 550  # paper's feedforward budget; use 2 for a quick workflow check
FAMILY = "feedforward"
CONFIGS = [
    {"name": "relu", "hidden_sizes": [64], "activations": ["relu"]},
    {"name": "relu_relu", "hidden_sizes": [64, 32], "activations": ["relu", "relu"]},
    {"name": "relu_sigmoid", "hidden_sizes": [64, 32], "activations": ["relu", "sigmoid"]},
]
MODEL_CLASS = FeedforwardNN


## Training and validation functions

The loss is categorical cross-entropy, implemented as negative log likelihood on the model's log probabilities. Gradients are clipped at norm 5. Each run resets the seed and data shuffle order for comparable conditions.


In [ ]:
loss_function = nn.NLLLoss()  # categorical cross-entropy on log probabilities


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, count = 0.0, 0, 0
    predictions, targets = [], []
    with torch.set_grad_enabled(training):
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            if training:
                optimizer.zero_grad()
            log_probs = model(X_batch)
            loss = loss_function(log_probs, y_batch)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
            predicted = log_probs.argmax(dim=1)
            total_loss += loss.item() * len(y_batch)
            correct += (predicted == y_batch).sum().item()
            count += len(y_batch)
            predictions.extend(predicted.detach().cpu().tolist())
            targets.extend(y_batch.detach().cpu().tolist())
    return {"loss": total_loss / count, "accuracy": correct / count,
            "macro_f1": f1_score(targets, predictions, labels=[0, 1, 2],
                                 average="macro", zero_division=0)}


def train_one(model, train_loader, val_loader):
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    history, best_state = [], None
    best_f1, best_loss, best_epoch, stale = -1.0, float("inf"), 0, 0
    for epoch in range(1, EPOCHS + 1):
        train = run_epoch(model, train_loader, optimizer)
        val = run_epoch(model, val_loader)
        history.append({"epoch": epoch,
                        "train_loss": train["loss"], "val_loss": val["loss"],
                        "train_accuracy": train["accuracy"], "val_accuracy": val["accuracy"],
                        "val_macro_f1": val["macro_f1"]})
        # Primary selection metric: validation macro F1; break ties by validation loss.
        improved = (val["macro_f1"] > best_f1 or
                    (val["macro_f1"] == best_f1 and val["loss"] < best_loss))
        if improved:
            best_f1, best_loss, best_epoch = val["macro_f1"], val["loss"], epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        if epoch == 1 or epoch % 25 == 0:
            print(f"  epoch {epoch}: train acc {train['accuracy']:.4f}; "
                  f"val acc {val['accuracy']:.4f}; val macro F1 {val['macro_f1']:.4f}")
        if EARLY_STOPPING and stale >= PATIENCE:
            break
    model.load_state_dict(best_state)
    return pd.DataFrame(history), best_epoch, run_epoch(model, val_loader)


## Run all experiments

This cell can take a long time, particularly on CPU. To check the workflow first, set `EPOCHS = 2` above. Return to the intended epoch budget and rerun before reporting project results.


In [ ]:
RESULTS = ROOT / "results" / FAMILY
RESULTS.mkdir(parents=True, exist_ok=True)
records = []
# Four embeddings x three configurations = 12 runs for this family.
for embedding_name in ["word2vec_50", "word2vec_100", "glove_50", "glove_100"]:
    for config in CONFIGS:
        run_id = f"{embedding_name}__{config['name']}"
        print("\nTraining", FAMILY, run_id)
        set_seed(SEED)
        train_loader = make_loader(X_train, y_train, shuffle=True)
        val_loader = make_loader(X_val, y_val)
        model = MODEL_CLASS(embeddings[embedding_name], config).to(DEVICE)
        started = time.time()
        history, best_epoch, val = train_one(model, train_loader, val_loader)
        settings = {"family": FAMILY, "embedding": embedding_name, "config": config,
                    "seed": SEED, "dropout": DROPOUT,
                    "trainable_embeddings": TRAINABLE_EMBEDDINGS,
                    "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE,
                    "epoch_budget": EPOCHS, "early_stopping": EARLY_STOPPING,
                    "patience": PATIENCE, "best_epoch": best_epoch,
                    "preparation_id": metadata["preparation_id"],
                    "word2vec_source": metadata["word2vec_source"],
                    "selection_metric": "validation macro F1; ties: lower validation loss",
                    "device": str(DEVICE),
                    "versions": {"python": platform.python_version(), "torch": str(torch.__version__),
                                 "numpy": np.__version__}}
        history.to_csv(RESULTS / f"{run_id}_history.csv", index=False)
        (RESULTS / f"{run_id}_settings.json").write_text(json.dumps(settings, indent=2))
        torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()},
                   RESULTS / f"{run_id}.pt")
        row = {"run_id": run_id, "family": FAMILY, "embedding": embedding_name,
               "config": config["name"], "best_epoch": best_epoch,
               "epochs_run": len(history), "val_accuracy": val["accuracy"],
               "val_macro_f1": val["macro_f1"], "val_loss": val["loss"],
               "seconds": round(time.time() - started, 1),
               "preparation_id": metadata["preparation_id"]}
        records.append(row)
        # Save after each run so completed results survive a later interruption.
        pd.DataFrame(records).to_csv(RESULTS / "validation_summary.csv", index=False)
        print(f"  best epoch {best_epoch}: val macro F1 {val['macro_f1']:.4f}")
        del model

summary = pd.DataFrame(records).sort_values(
    ["val_macro_f1", "val_loss", "run_id"], ascending=[False, True, True])
display(summary.drop(columns=["preparation_id"]))


## Preview the validation winner


In [ ]:
# Inspect the validation winner; every run's complete history is saved for notebook 04.
best_run = summary.iloc[0]["run_id"]
history = pd.read_csv(RESULTS / f"{best_run}_history.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for split, label in [("train", "Training"), ("val", "Validation")]:
    axes[0].plot(history["epoch"], history[f"{split}_loss"], label=label)
    axes[1].plot(history["epoch"], history[f"{split}_accuracy"], label=label)
for ax, ylabel in zip(axes, ["Categorical cross-entropy", "Accuracy"]):
    ax.set(xlabel="Epoch", ylabel=ylabel)
    ax.legend()
    ax.grid(alpha=0.2)
fig.suptitle(f"{FAMILY}: {best_run}")
fig.tight_layout()
plt.show()
print("Completed", len(summary), "experiments. Test data has not been evaluated.")
